## Data source:

Website: 'https://www.salsify.com/'

## Data preprocessing

#### Step 1: We created features_en.json, a cleaned dataset read by our clustering notebooks, and products_en.json, for evaluation and later business-related uses. The first contains attribute_records, with one entry per unique feature title. The latter contains products, with the feature titles for each product id.

In [4]:
import html
import json
from pathlib import Path

#input
catalog_path = Path("data/catalog.json")
catalog = json.loads(catalog_path.read_text(encoding="utf8"))  # changed: load the catalog

#output
features_path = Path("data/features_en.json")
products_path = Path("data/products_en.json")

#collecting text blocks
blocks = []
for product in catalog["products"]:
    product_type = product["data"]["classification"]["product_type"]
    blocks.append((product["id"], product_type, product["data"]["content"].get("en-GB", {})))
    for variant in product.get("variants", []):  # changed: products without variants
        content = (variant.get("overrides") or {}).get("content", {}).get("en-GB", {})
        blocks.append((product["id"], product_type, content))

#for feature_en.json, will contain one entry per unique title and holds its descriptions 
records = {}

#for products_en.json, will contain one entry per product and holds its titles
products = {}

#for feature_en.json, at the end of the file, for descriptions that cant be matched to a title
unpaired = []

for product_id, product_type, content in blocks:
    titles = [html.unescape(t).strip() for t in content.get("feature_titles") or []]
    descs = [html.unescape(d).strip() for d in content.get("features") or []] 

    product = products.setdefault(product_id, {"product_id": product_id, "product_type": product_type, "titles": set()})
    product["titles"].update(titles)

    for title in titles:
        records.setdefault(title, {"title": title, "descriptions": []})

    # Titles and descriptions are only linked by position through zip. If the counts differ
    # we can't tell which title is missing a description, so we don't pair them and include into unpaired.
    if len(titles) != len(descs):
        unpaired += descs
        continue
    for title, desc in zip(titles, descs):
        records[title]["descriptions"].append(desc)

for product in products.values():
    product["titles"] = sorted(product["titles"])


features_path.write_text(
    json.dumps({"attribute_records": list(records.values()), "unpaired_descriptions": unpaired}, ensure_ascii=False, indent=1),
    encoding="utf8",
)

products_path.write_text(
    json.dumps(list(products.values()), ensure_ascii=False, indent=1),
    encoding="utf8",
)
print(len(records), "unique titles")
print(len(unpaired), "unpaired descriptions")
print(len(products), "products")

445 unique titles
22 unpaired descriptions
208 products
